In [ ]:
!pip install deep_translator
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.8 MB/s eta 0:00:00


# Introducción

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer, util
from deep_translator import GoogleTranslator
from transformers import BertTokenizer, BertModel
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer
import requests
from bs4 import BeautifulSoup
import regex as re
from gliner import GLiNER
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import ast
from nltk.stem import SnowballStemmer
nltk.download('stopwords')
nltk.download('punkt')


/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# They have no header, the columns are separated by a semicolon and should be named "frase" and "animo"
df = pd.read_csv('emociones.csv', sep=',', header=0, names=['frase', 'animo'])


In [ ]:
df.head()

,frase,animo
0,Hoy es el día más feliz de mi vida,feliz
1,Conseguí el trabajo de mis sueños,feliz
2,Mi hijo dio sus primeros pasos,feliz
3,Acabo de casarme con el amor de mi vida,feliz
4,Gané el primer premio en el concurso,feliz


# Análisis exploratorio

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 454 entries, 0 to 453
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   frase   454 non-null    object
 1   animo   454 non-null    object
dtypes: object(2)
memory usage: 7.2+ KB


Vemos que no tenemos ningún dato faltante. Acomodemos las categorizaciones de las frases a las deseadas: "Alegre", "Desanimado/Apático" y "Neutral"

In [ ]:
df["animo"].unique()

array(['feliz', 'neutral', 'triste'], dtype=object)

In [ ]:
# translate to english
trad = GoogleTranslator(source='es', target='en')
df["frase"] = df["frase"].apply(lambda x: trad.translate(x).lower())
X = df['frase']
y = df['animo']

In [ ]:
df.to_csv('emociones_en.csv', index=False)

In [ ]:
df = pd.read_csv('emociones_en.csv')
X = df['frase']
y = df['animo']

In [ ]:
# # Cargamos el modelo desde HuggingFace https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

X_vectorized = model.encode(X.reset_index(drop=True))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# División en conjunto de entrenamiento y prueba
X_train_vectorized, X_test_vectorized, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

In [ ]:
X_train_vectorized

array([[ 0.02871957,  0.09351845,  0.01126155, ...,  0.03485187,
         0.01970048,  0.00582241],
       [ 0.00352777,  0.0545264 ,  0.00098997, ...,  0.00044855,
         0.03971175, -0.00933841],
       [ 0.0181774 ,  0.12084132, -0.01312581, ...,  0.02836607,
        -0.01514078, -0.02439292],
       ...,
       [-0.03184416, -0.0079573 , -0.02802826, ...,  0.01827721,
        -0.04282591,  0.0138965 ],
       [ 0.01155355,  0.08603133,  0.0238132 , ..., -0.03629459,
         0.01689411, -0.0072225 ],
       [ 0.01058322,  0.01464105, -0.02692855, ...,  0.00601015,
         0.01754897, -0.00540267]], dtype=float32)

# Entrenando clasificador de ánimo

In [ ]:
# Entrenamiento del modelo
classifier_animo = LogisticRegression(multi_class='multinomial', solver='lbfgs')
classifier_animo.fit(X_train_vectorized, y_train)

# Evaluación inicial
y_pred = classifier_animo.predict(X_test_vectorized)
acc_LR = accuracy_score(y_test, y_pred)
report_LR = classification_report(y_test, y_pred, zero_division=1)


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
print("Precisión Regresión Logística:", acc_LR)
print("Reporte de clasificación Regresión Logística:\n", report_LR)

Precisión Regresión Logística: 0.978021978021978
Reporte de clasificación Regresión Logística:
               precision    recall  f1-score   support

       feliz       0.97      1.00      0.98        29
     neutral       0.97      0.97      0.97        33
      triste       1.00      0.97      0.98        29

    accuracy                           0.98        91
   macro avg       0.98      0.98      0.98        91
weighted avg       0.98      0.98      0.98        91



# Obteniendo datos de juegos de mesa, libros y películas

Los juegos de mesa y películas están en archivos .csv

In [ ]:
df_juegos_mesa = pd.read_csv('bgg_database.csv')
df_peliculas = pd.read_csv('IMDB-Movie-Data.csv')

In [ ]:
df_peliculas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Rank                1000 non-null   int64  
 1   Title               1000 non-null   object 
 2   Genre               1000 non-null   object 
 3   Description         1000 non-null   object 
 4   Director            1000 non-null   object 
 5   Actors              1000 non-null   object 
 6   Year                1000 non-null   int64  
 7   Runtime (Minutes)   1000 non-null   int64  
 8   Rating              1000 non-null   float64
 9   Votes               1000 non-null   int64  
 10  Revenue (Millions)  1000 non-null   float64
 11  Metascore           1000 non-null   int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 93.9+ KB


In [ ]:
df_juegos_mesa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   rank              1000 non-null   int64  
 1   game_name         1000 non-null   object 
 2   game_href         1000 non-null   object 
 3   geek_rating       1000 non-null   float64
 4   avg_rating        1000 non-null   float64
 5   num_voters        1000 non-null   float64
 6   description       1000 non-null   object 
 7   yearpublished     1000 non-null   int64  
 8   minplayers        1000 non-null   int64  
 9   maxplayers        1000 non-null   int64  
 10  minplaytime       1000 non-null   int64  
 11  maxplaytime       1000 non-null   int64  
 12  minage            1000 non-null   int64  
 13  avgweight         1000 non-null   float64
 14  best_num_players  1000 non-null   object 
 15  designers         1000 non-null   object 
 16  mechanics         1000 non-null   object 
 

Eliminamos todas las columnas que no nos interesan

In [ ]:

df_juegos_mesa = df_juegos_mesa[["game_name", "description", "categories"]].rename(columns={"game_name": "Titulo", "description": "Descripcion", "categories": "Genero"})
df_peliculas = df_peliculas[["Title", "Description", "Genre"]].rename(columns={"Title": "Titulo", "Description": "Descripcion", "Genre": "Genero"})
# Usados luego para mostrar respuestas al usuario
df_juegos_mesa_original = df_juegos_mesa.copy()
df_peliculas_original = df_peliculas.copy()

## Web scraping para los datos de libros

In [ ]:
# URL of the Gutenberg Top 1000 books page
url = "https://www.gutenberg.org/browse/scores/top1000.php#books-last1"

# Send a GET request to the page
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

In [ ]:
index = 0

# Find and print all book titles in the "Most Popular Books" section
datos_libros = []

for link in soup.select('ol li a'):
    resumen_hallado = False
    # obtenemos el libro
    titulo_libro = link.get_text()

    # Obtenemos la descripción
    href = link['href']
    url_libro = "https://www.gutenberg.org" + href
    response_libro = requests.get(url_libro)
    soup_libro = BeautifulSoup(response_libro.content, 'html.parser')
    # Find the table rows

    subjects = ""

    # Loop through table rows to find "Summary" and "Subject"
    for row in soup_libro.select('table tr'):
        header = row.find('th')
        data = row.find('td')

        if header and data:
            if "Summary" in header.get_text():
                resumen = data.get_text(strip=True)
                resumen_hallado = True
            elif "Subject" in header.get_text():
                subjects += data.get_text(strip=True) + " "
    genero = subjects

    datos_libros.append({"Titulo": titulo_libro,
                         "Resumen": (resumen if resumen_hallado else None),
                         "Genero": genero})

    if index % 25 == 0:
        print(f"Libro {index}")
    index += 1
    if index == 1000:
        break
df_libros = pd.DataFrame(datos_libros).rename(columns={"Resumen": "Descripcion"})

Libro 0
Libro 25
Libro 50
Libro 75
Libro 100
Libro 125
Libro 150
Libro 175
Libro 200
Libro 225
Libro 250
Libro 275
Libro 300
Libro 325
Libro 350
Libro 375
Libro 400
Libro 425
Libro 450
Libro 475
Libro 500
Libro 525
Libro 550
Libro 575
Libro 600
Libro 625
Libro 650
Libro 675
Libro 700
Libro 725
Libro 750
Libro 775
Libro 800
Libro 825
Libro 850
Libro 875
Libro 900
Libro 925
Libro 950
Libro 975


In [ ]:
# como lo anterior es un proceso que toma bastante tiempo, lo guardamos para no tener que repetirlo cada vez
df_libros.to_csv('libros.csv', index=False)

In [ ]:
df_libros = pd.read_csv('libros.csv', dtype={'Titulo': str, 'Resumen': str, 'Genero': str})

NameError: name 'datos_libros' is not defined

# Limpieza de datos

In [ ]:
df_libros.fillna("", inplace=True)
# Lo usamos para mantener la información en un formato que luego pueda ser mostrado al usuario
df_libros.rename(columns={"Resumen": "Descripcion"}, inplace=True)
df_libros_original = df_libros.copy()

### Curando la categoría géneros para todos los datos


In [ ]:
stemmer = SnowballStemmer("english")

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_text = [stemmer.stem(word) for word in word_tokens if word.casefold() not in stop_words]
    return " ".join(filtered_text)

df_libros["Genero"] = df_libros["Genero"].apply(remove_stopwords)

In [ ]:
def juegos_genero_stemms(generos):
    genero_str = generos[1:-1].replace("'", "").replace(",", " ")
    return stemmer.stem(genero_str)

df_juegos_mesa["Genero"] = df_juegos_mesa["Genero"].apply(juegos_genero_stemms)

In [ ]:
df_juegos_mesa["Genero"] = df_juegos_mesa["Genero"].apply(remove_stopwords)
df_peliculas["Genero"] = df_peliculas["Genero"].apply(remove_stopwords)

# Ponderador de actividad a recomendar con clasificador

Entrenaremos una regresión logística, no para predecir una clase, sino que la usaremos para predecir probabilidades, y con esas probabilidades premiar ciertas actividades mas que otras.

In [ ]:
# They have no header, the columns are separated by a semicolon and should be named "frase" and "animo"
df_actividad = pd.read_csv('actividad.csv', sep=',', header=0, names=['texto', 'preferencia'])

df_actividad["preferencia"].value_counts()

,count
preferencia,
pelicula,84
juegomesa,84
libro,84


In [ ]:
df_actividad["texto"] = df_actividad["texto"].apply(lambda x: trad.translate(x).lower())
df_actividad.to_csv('actividad_en.csv', index=False)

In [ ]:
df_actividad = pd.read_csv('actividad_en.csv')

In [ ]:
X = df_actividad['texto']
y = df_actividad['preferencia']

In [ ]:
# # Cargamos el modelo desde HuggingFace https://huggingface.co/sentence-transformers/all-mpnet-base-v2
embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

X_vectorized = model.encode(X.reset_index(drop=True))

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# División en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

In [ ]:
# Entrenamiento del modelo
classifier_act = LogisticRegression(multi_class='multinomial', solver='lbfgs')
classifier_act.fit(X_train, y_train)

# Evaluación inicial
y_pred = classifier_act.predict(X_test)
acc_LR = accuracy_score(y_test, y_pred)
report_LR = classification_report(y_test, y_pred, zero_division=1)


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
print("Precisión Regresión Logística:", acc_LR)
print("Reporte de clasificación Regresión Logística:\n", report_LR)

Precisión Regresión Logística: 0.9803921568627451
Reporte de clasificación Regresión Logística:
               precision    recall  f1-score   support

   juegomesa       1.00      1.00      1.00        19
       libro       0.91      1.00      0.95        10
    pelicula       1.00      0.95      0.98        22

    accuracy                           0.98        51
   macro avg       0.97      0.98      0.98        51
weighted avg       0.98      0.98      0.98        51



In [ ]:
# df con todos los datos para búsqueda
df_todos_libros = df_libros.copy()
df_todos_libros["Tipo"] = "libro"
df_todos_juegos_mesa = df_juegos_mesa.copy()
df_todos_juegos_mesa["Tipo"] = "juegomesa"
df_todos_peliculas = df_peliculas.copy()
df_todos_peliculas["Tipo"] = "pelicula"

# Para mostrar datos de las recomendaciones al usuario
dfs_originales_peliculas = df_peliculas_original.copy()
dfs_originales_peliculas["Tipo"] = "Película"
dfs_originales_libros = df_libros_original.copy()
dfs_originales_libros["Tipo"] = "Libro"
dfs_originales_juegos_mesa = df_juegos_mesa_original.copy()
dfs_originales_juegos_mesa["Tipo"] = "Juego de mesa"


dfs_originales = pd.concat([dfs_originales_libros, dfs_originales_juegos_mesa, dfs_originales_peliculas]).reset_index(drop=True)
df_prediccion = pd.concat([df_todos_libros, df_todos_juegos_mesa, df_todos_peliculas]).reset_index(drop=True)

In [ ]:
df_prediccion.head()

,Titulo,Descripcion,Genero,Tipo
0,"Frankenstein; Or, The Modern Prometheus by Mar...","""Frankenstein; Or, The Modern Prometheus"" by M...",scienc fiction horror tale gothic fiction scie...,libro
1,呻吟語 by Kun Lü (3588),"""呻吟語"" by Kun Lü is a philosophical treatise wr...",conduct life,libro
2,Pride and Prejudice by Jane Austen (3274),"""Pride and Prejudice"" by Jane Austen is a clas...",england -- fiction young women -- fiction love...,libro
3,"Moby Dick; Or, The Whale by Herman Melville (2...","""Moby Dick; Or, The Whale"" by Herman Melville ...",whale -- fiction sea stori psycholog fiction s...,libro
4,The Scarlet Letter by Nathaniel Hawthorne (2343),"""The Scarlet Letter"" by Nathaniel Hawthorne is...",adulteri -- fiction histor fiction reveng -- f...,libro


In [ ]:
# Usados para la comparación semántica, usarlos dentro del dataframe sería incómodo
embeddings_descripcion = model.encode(df_prediccion["Descripcion"].reset_index(drop=True))

In [ ]:
# Guardamos embeddings
np.save('embeddings_descripcion.npy', embeddings_descripcion)

In [ ]:
embeddings_descripcion = np.load('embeddings_descripcion.npy')

# Ner y coincidencia semántica

In [ ]:
# Carga el modelo preentrenado 'gliner_multi-v2.1' desde Hugging Face
model_gliner = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

# Cambia el modelo a modo de evaluación, esto es útil para desactivar características específicas como dropout durante la inferencia
model_gliner.eval()


# Lista de etiquetas que el modelo intentará encontrar en el título
labels = ["author", "actor", "character", "saga"]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

gliner_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.77k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
# obtenemos todas las entidades de todos los títulos, pues solo los compararemos a través de sus entidades
titulos_ner = df_prediccion["Titulo"].apply(lambda x: model_gliner.predict_entities(x, labels, threshold=0.4))
titulos_ner.reset_index(drop=True, inplace=True)
df_prediccion["Titulo"] = titulos_ner
df_prediccion.drop(columns=["Descripcion"], inplace=True)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
titulos_ner.to_csv("titulos_ner.csv", index=False)

In [ ]:
titulos_ner = pd.read_csv("titulos_ner.csv")

# titulos_ner.reset_index(drop=True, inplace=True)
# df_prediccion["Titulo"] = titulos_ner
# df_prediccion.drop(columns=["Descripcion"], inplace=True)

# Funciones para procesar recomendaciones

In [ ]:
rows_juegos = df_prediccion["Tipo"] == "juegomesa"
rows_libros = df_prediccion["Tipo"] == "libro"
rows_peliculas = df_prediccion["Tipo"] == "pelicula"


def considerar_preferencia_medio(preferencia):
    df_prediccion.loc[rows_juegos, "puntaje"] = preferencia[0]
    df_prediccion.loc[rows_libros, "puntaje"] = preferencia[1]
    df_prediccion.loc[rows_peliculas, "puntaje"] = preferencia[2]

Si el usuario siente ciertas emociones, será menos o más deseable recomendar algunos tipos particulares de juegos, libros o películas.

Por ejemplo, si el usuario está triste, probablemente no querrá ver una película de amor, guerra o drama, pero sí una de comedia.

Primero que nada, haremos stemming sobre los géneros que queremos filtrar para facilitar la busqueda de coincidencias.


In [ ]:

stemmer = SnowballStemmer("english")
#generos que no queremos recomendar a alguien triste o melancólico
generos_triste = ["war","romance","love","drama","police","horror"]
#generos que no queremos recomendar a alguien alegre
generos_feliz = ["dictionary","","documentary","translation","history","instruction","manual", "text"]

# Aplicamos la derivación a cada palabra en el texto
stemmed_triste = [stemmer.stem(word) for word in generos_triste]
stemmed_feliz = [stemmer.stem(word) for word in generos_feliz]

palabras_no_triste = "|".join(stemmed_triste)
palabras_no_feliz = "|".join(stemmed_feliz)

In [ ]:
# Queremos penalizar ciertas generos en base al ánimo
def considerar_animo(animo):
    if animo == "feliz":
        return df_prediccion["Genero"].apply(lambda x: -0.4 if re.search(palabras_no_feliz, x, flags=re.IGNORECASE) else 0)
    elif animo == "triste":
        return df_prediccion["Genero"].apply(lambda x: -0.4 if re.search(palabras_no_triste, x, flags=re.IGNORECASE) else 0)

In [ ]:

# Favorecemos aquellos títulos cuyas entidades coincidan con las halladas en el prompt del usuario
def comparar_entidades(entidades_consulta, titulo):
    dicts = ast.literal_eval(str(titulo))
    entidades_titulo = dict()
    coincidencias = 0
    for diccionario in dicts:
        entidades_titulo[diccionario["label"]] = set()

    for diccionario in dicts:
        entidades_titulo[diccionario["label"]].add(diccionario["text"])

    for key, value in entidades_consulta.items():
        if key in entidades_titulo:
            coincidencias += len(value.intersection(entidades_titulo[key]))
    return float(coincidencias / 2)

def considerar_entidades(entidades_consulta):
    df_prediccion["puntaje"] += df_prediccion["Titulo"].apply(lambda x: comparar_entidades(entidades_consulta, x))

In [ ]:
def considerar_descripcion(incrustacion_consulta):
    df_prediccion["puntaje"] = df_prediccion["puntaje"] + pd.Series(util.cos_sim(embeddings_descripcion, incrustacion_consulta).flatten())


# Recomendaciones

In [ ]:
# # Cargamos el modelo desde HuggingFace https://huggingface.co/sentence-transformers/all-mpnet-base-v2
trad = GoogleTranslator(source='es', target='en')
embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
def recomendar():

    df_prediccion["puntaje"] = 0
    prompt_animo = input("¿Cómo te sientes hoy?\n")

    incrustacion_animo = embedder.encode(trad.translate(prompt_animo))
    prediccion_animo = classifier_animo.predict(incrustacion_animo.reshape(1, -1))[0]

    # Penalizamos aquellos géneros que consideramos poco aptos para el estado de ánimo
    considerar_animo(prediccion_animo)

    # Prompt del usuario
    consulta = input("¿Cuál es tu interés en este momento?\n")
    # Generar incrustaciones para todas las consultas y respuestas
    consulta_en = trad.translate(consulta).lower()
    incrustacion_consulta = embedder.encode(trad.translate(consulta_en))

    # Vemos qué tipo de actividad favorecemos en la recomendación
    preferencia_medio = classifier_act.predict_proba(embedder.encode([consulta_en]))[0]
    # preferencia_medio[0] = juegomesa, preferencia_medio[1] = Libro, preferencia_medio[2] = pelicula

    df_prediccion["puntaje"] = 0
    considerar_preferencia_medio(preferencia_medio)

    # obtenemos las entidades del prompt del usuario
    entidades_consulta = dict()

    entidades = model_gliner.predict_entities(consulta, labels, threshold=0.4)
    for diccionario in entidades:
        entidades_consulta[diccionario["label"]] = set()

    for diccionario in entidades:
        entidades_consulta[diccionario["label"]].add(diccionario["text"])
    considerar_entidades(entidades_consulta)

    # Comparacion semántica con las incrustaciones de descripciones
    considerar_descripcion(incrustacion_consulta)

    # Devolvemos los índices, para poder ubicar la información en el
    # dataframe con la información correctamente formateada
    return df_prediccion.nlargest(5, "puntaje")


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
traductor_en_a_es = GoogleTranslator(source='en', target='es')
def printear_recomendaciones(recomendaciones):
    print("Te recomendamos las siguientes actividades:\n")
    for row in recomendaciones.index:
        print(f"Título: {recomendaciones.loc[row, 'Titulo']}")
        print(f"Tipo: {recomendaciones.loc[row, 'Tipo']}")
        if pd.isna(recomendaciones.loc[row, "Descripcion"]):
            print(f"Descripción: No se encuentra disponible una descripcion para esta actividad")
        else:
            print(f"Descripción: {traductor_en_a_es.translate(recomendaciones.loc[row, 'Descripcion'])}")
        print(f"Genero: {traductor_en_a_es.translate(recomendaciones.loc[row, 'Genero'])}")
        print("")

In [ ]:
# Se pide un prompt y se recomienda en consecuencia
recomendaciones = dfs_originales.loc[recomendar().index]
printear_recomendaciones(recomendaciones)

¿Cómo te sientes hoy?
Hace días que no sale el sol, me aburro mucho en casa.
¿Cuál es tu interés en este momento?
Tengo ganas de leer una buena historia de fantasía medieval.


<ipython-input-82-eba772a89d2e>:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.14911187049394253' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_prediccion.loc[rows_juegos, "puntaje"] = preferencia[0]


Te recomendamos las siguientes actividades:

Título: The Thousand and One Nights, Vol. I. (141)
Tipo: Libro
Descripción: "Las mil y una noches, vol. I" de Edward William Lane es una colección de cuentos populares de Oriente Medio escritos en el siglo XIX. Este clásico literario, a menudo conocido como "Las mil y una noches", abarca una variedad de historias encantadoras, enmarcadas en la narrativa de Shahrazád, quien cuenta cuentos para cautivar al rey Shahriyár. El volumen sirve como una rica exploración de temas como el amor, la traición y la astucia, todo ello en un contexto de opulenta cultura árabe. Al comienzo de la colección, los lectores aprenden sobre el rey Shahriyár y su hermano Shah-Zemán, quienes han gobernado sus reinos con justicia y alegría durante veinte años. Sus vidas dan un giro oscuro cuando ambos descubren infidelidades de sus esposas, lo que conduce a acciones devastadoras. Esto prepara el escenario para Shahrazád, la hija del visir, que se ofrece voluntaria para